# Notebook 02: Silver Transfortation
## Propósito
Leer tablas Bronze, aplicar limpieza, casting de tipo, deduplicación, campos calculados y persistencia en Silver
## Inputs
- 8 tablas "bronze_*" en {BRONZE_LAKEHOUSE}
## Outputs
- 7 tablas "silver_* en {SILVER_LAKEHOUSE} (se combina productos con categoria) (product + translation)
## Cross-lakehouse pattern
- Reads: spark.read.tables(f"{BRONZE_LAKEHOUSE}.{tabla}")
- Writes: df.write.saveAsTable(f"{SILVER_LAKEHOUSE}.{tabla}")


In [6]:
# =================================================================
# PARÁMETROS DEL NOTEBOOK
# =================================================================
# Estos valores se pueden sobrescribir cuando el notebook es invocado
# desde un Data Pipeline. Si se corre manualmente, usa estos defaults.
# -----------------------------------------------------------------

#Lakehouse de origen (Bronze) - solo lectura
BRONZE_LAKEHOUSE = "lh_olist_bronze"

#Lakehouse de destino (Silver)
SILVER_LAKEHOUSE = "lh_olist_silver"

#Modo de escritura
WRITE_MODE = "overwrite"

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 8, Finished, Available, Finished, False)

In [7]:
# **Celda 3 — Imports:**
from pyspark.sql.functions import (col, lower, trim, when, coalesce, 
                                    to_timestamp, datediff, length, 
                                    round as spark_round, current_timestamp, lit)
from pyspark.sql.types import DoubleType, IntegerType

print(f"Bronze (source): {BRONZE_LAKEHOUSE}")
print(f"Silver (target): {SILVER_LAKEHOUSE}")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 9, Finished, Available, Finished, False)

Bronze (source): lh_olist_bronze
Silver (target): lh_olist_silver


In [8]:
# **Celda 4 — Helpers:**

def read_bronze(table_name: str):
    "Lee una tabla desde el lakehouse de Bronze"
    return spark.read.table(f"{BRONZE_LAKEHOUSE}.{table_name}")

def write_silver(df, table_name: str):
    "Escribe un dataframe como tabla Delta en Silver"
    (df.write.format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_LAKEHOUSE}.{table_name}"))
    print(f"{SILVER_LAKEHOUSE}-{table_name}: {df.count():,} filas")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 10, Finished, Available, Finished, False)

In [9]:
# **Celda 5 — Silver: customers:**

print("silver_customer")

silver_customers = (read_bronze("bronze_customers")
    .dropDuplicates(["customer_id"])
    .withColumn("customer_city", lower(trim(col("customer_city"))))
    .withColumn("customer_state", trim(col("customer_state")))
    .drop("_ingestion_timestamp", "_source_file"))

write_silver(silver_customers, "silver_customers")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 11, Finished, Available, Finished, False)

silver_customer
lh_olist_silver-silver_customers: 99,441 filas


In [13]:
# **Celda 6 — Silver: orders:**

print("silver_orders")

silver_orders = (read_bronze("bronze_orders")
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date"))
    .withColumn("approval_lag_hours", (col("order_approved_at").cast("long") - col("order_purchase_timestamp").cast("long")) / 3600)
    .withColumn("delivery_delay_days", datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))
    .withColumn("delivery_status_bucket", 
        when(col("order_status") != "delivered", "Not Delivered")
        .when(col("delivery_delay_days") <= 0, "On time")
        .when(col("delivery_delay_days") <= 7, "Late <=7d")
        .otherwise("Late >7d"))
    .drop("_ingestion_timestamps", "_source_file"))

write_silver(silver_orders, "silver_orders")



StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 15, Finished, Available, Finished, False)

silver_orders
lh_olist_silver-silver_orders: 99,441 filas


In [14]:
# **Celda 7 — Silver: order_items:**

print("silver_order_items")

silver_order_items = (read_bronze("bronze_order_items")
    .withColumn("price", col("price"). cast(DoubleType()))
    .withColumn("freight_value", col("freight_value").cast(DoubleType()))
    .withColumn("shipping_limit_date", to_timestamp("shipping_limit_date"))
    .withColumn("total_item_value", spark_round(col("price") + col("freight_value"), 2))
    .drop("_ingestion_timestamp", "_source_file"))

write_silver(silver_order_items, "silver_order_items")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 16, Finished, Available, Finished, False)

silver_order_items
lh_olist_silver-silver_order_items: 112,650 filas


In [16]:
# **Celda 8 — Silver: order_payments:**

print("silver_order_payments")

silver_order_payments = (read_bronze("bronze_order_payments")
    .withColumn("payment_value", col("payment_value").cast(DoubleType()))
    .withColumn("payment_installments", col("payment_installments").cast(IntegerType())) #cuota de pago
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn("installments_bucket",
        when(col("payment_installments") == 1, "1")
        .when(col("payment_installments") <= 5, "2-5")
        .when(col("payment_installments") <= 12, "6-12")
        .otherwise("13+"))
    .drop("_ingestion_timestamp", "_source_file"))

write_silver(silver_order_payments, "silver_order_payments")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 18, Finished, Available, Finished, False)

silver_order_payments
lh_olist_silver-silver_order_payments: 103,886 filas


In [19]:
# **Celda 9 — Silver: order_reviews:**

print("silver_order_reviews")

silver_order_reviews = (read_bronze("bronze_order_reviews")
    .dropDuplicates(["review_id"])
    .withColumn("review_score", col("review_score"). cast(IntegerType()))
    .withColumn("review_creation_date", to_timestamp("review_creation_date"))
    .withColumn("review_answer_timestamp", to_timestamp("review_answer_timestamp"))
    .withColumn("response_time_hours",
        (col("review_answer_timestamp").cast("long") - col("review_creation_date").cast("long")) / 3600)
    .withColumn("has_comment",
        when(col("review_comment_message").isNotNull(), True).otherwise(False))
    .withColumn("comment_lenght",
        coalesce(length(col("review_comment_message")), lit(0)))
    .drop("_ingestion_timestamp", "_source_file"))

write_silver(silver_order_reviews, "silver_order_reviews")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 21, Finished, Available, Finished, False)

silver_order_reviews
lh_olist_silver-silver_order_reviews: 98,410 filas


In [20]:
# **Celda 10 — Silver: products (con categoria):**

print("silver_products (con join de traducción)")

bronze_products = read_bronze("bronze_products")
bronze_translation = read_bronze("bronze_category_translation")

silver_products = (bronze_products
    .dropDuplicates(["product_id"])
    .join(bronze_translation, on="product_category_name", how="left")
    .withColumn("product_category_name_english",
        coalesce(col("product_category_name_english"), lit("unknown")))
    .withColumn("product_weight_g",   col("product_weight_g").cast(DoubleType()))
    .withColumn("product_length_cm",  col("product_length_cm").cast(DoubleType()))
    .withColumn("product_height_cm",  col("product_height_cm").cast(DoubleType()))
    .withColumn("product_width_cm",   col("product_width_cm").cast(DoubleType()))
    .drop("_ingestion_timestamp", "_source_file"))

write_silver(silver_products, "silver_products")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 22, Finished, Available, Finished, False)

silver_products (con join de traducción)
lh_olist_silver-silver_products: 32,951 filas


In [21]:
# **Celda 11 — Silver: sellers:**

print("silver_sellers")

silver_sellers = (read_bronze("bronze_sellers")
    .dropDuplicates(["seller_id"])
    .withColumn("seller_city",  lower(trim(col("seller_city"))))
    .withColumn("seller_state", trim(col("seller_state")))
    .drop("_ingestion_timestamp", "_source_file"))

write_silver(silver_sellers, "silver_sellers")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 23, Finished, Available, Finished, False)

silver_sellers
lh_olist_silver-silver_sellers: 3,095 filas


In [22]:

# **Celda 12 — Validación de Silver:**

print("=" * 60)
print("VALIDACIÓN SILVER")
print("=" * 60)

TABLAS_SILVER = {
    "silver_customers":"customer_id",
    "silver_orders":"order_id",
    "silver_order_items":"order_id",
    "silver_order_payments":"order_id",
    "silver_order_reviews":"review_id",
    "silver_products":"product_id",
    "silver_sellers":"seller_id",
}

problemas = 0
for t, pk in TABLAS_SILVER.items():
    df = spark.read.table(f"{SILVER_LAKEHOUSE}.{t}")
    count = df.count()
    nulls_pk = df.filter(col(pk).isNull()).count()
    ok = count > 0 and nulls_pk == 0
    status = "correcto" if ok else "incorrecto"
    print(f"{status} {t}: {count:,} filas, {nulls_pk} nulls en PK")
    if not ok:
        problemas += 1

if problemas:
    raise Exception(f"Validación Silver fallo: {problemas} problemas")
print("Silver completo")

StatementMeta(, ae6f1c9c-d376-4f32-a1dd-51bed094e035, 24, Finished, Available, Finished, False)

VALIDACIÓN SILVER
correcto silver_customers: 99,441 filas, 0 nulls en PK
correcto silver_orders: 99,441 filas, 0 nulls en PK
correcto silver_order_items: 112,650 filas, 0 nulls en PK
correcto silver_order_payments: 103,886 filas, 0 nulls en PK
correcto silver_order_reviews: 98,410 filas, 0 nulls en PK
correcto silver_products: 32,951 filas, 0 nulls en PK
correcto silver_sellers: 3,095 filas, 0 nulls en PK
Silver completo
